# Step 1: Setup & Data Preparation

In [9]:
# Cài đặt các thư viện Deep Learning cần thiết
# Lưu ý: Nếu chạy trên Terminal/CMD thì bỏ dấu '!' đi
#!pip install pandas numpy scikit-learn torch transformers datasets


In [10]:
import torch
print("Có CUDA không?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tên Card:", torch.cuda.get_device_name(0))
    print("Phiên bản CUDA:", torch.version.cuda)
else:
    print("Vẫn đang dùng CPU.")

Có CUDA không?: True
Tên Card: NVIDIA GeForce GTX 1650
Phiên bản CUDA: 11.8


In [11]:
# 1. Load dữ liệu đã gán(SEED DATA) 
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Thiết bị đang sử dụng: {device}")


 Thiết bị đang sử dụng: cuda


In [12]:
# Load dữ liệu đã gán nhãn (424 dòng chuẩn)
df_labeled = pd.read_csv("training_data.csv")
#df_labeled.head()
#df_labeled.shape
file_path_unlabeled = 'unlabeled_data.csv'
df_unlabeled = pd.read_csv(file_path_unlabeled)
#df_unlabeled.shape    
df_unlabeled.head()

,id,text
0,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzg1Njk1MDM0MD...,"<person> thôi ra hp còn đỡ, mà mình hay ru..."
1,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE1MzQ0MTQ3NT...,tôi tận hưởng rồi xoài 1/² kg tôi tưởng một ký...
2,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzE5MjQyNTMxNj...,<person> thuyệt tình à chứ
3,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0Xzc3NTMxNDMxOD...,thà đi nước ngoài
4,Y29tbWVudDoxMjcyNTYxNjkxNTY5MDU0XzEwMTU5MDU3NT...,"<person> tin chuẩn chưa bạn yêu, nghèo có được..."


NameError: name 'tokenizer' is not defined

In [15]:
# Chuẩn bị class dataset cho pytorch
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        '''
        item = {
            'input_ids': Tensor([10, 20]),
            'attention_mask': Tensor([1, 1])
        }
        '''
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings["input_ids"])
# tokenizer dữ liệu (biến chữ thành số)
model_name = "vinai/phobert-base"   
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_labeled["text"].tolist(),
    df_labeled["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_labeled["label"].tolist()
)

train_encodings = tokenizer(train_texts, padding=True, truncation= True, max_length=128)
val_encodings = tokenizer(val_texts, padding=True, truncation = True, max_length = 128)

train_dataset = HateSpeechDataset(train_encodings, train_labels)
val_dataset = HateSpeechDataset(val_encodings, val_labels)


In [ ]:
print(f"1. Số lượng tập Train: {len(train_texts)} dòng")
print(train_texts[:5])
print(f"\n2. Số lượng tập Validation: {len(val_texts)} dòng")
print(val_texts[:5])
# Xem thử mẫu đầu tiên trông như thế nào
print("\n--- Mẫu dữ liệu đầu tiên trong tập Train ---")
print(f"Câu văn (Text): {train_texts[0]}")
print(f"Nhãn (Label):   {train_labels[0]} (0=Không ghét, 1=Ghét)")



# Giả sử bạn có 400 dòng dữ liệu
print(train_encodings['input_ids'].shape)
# Kết quả sẽ là: torch.Size([400, 128])
# Nghĩa là: 400 dòng, 128 cột.


1. Số lượng tập Train: 339 dòng
['gửi clip này cho công an  giao thông điều tra để xử phạt vì phạm pháp luật !', 'thằng này không ra gì,', '<person> làm giàu thì mới có tiền đi từ thiện ngu', 'quy vào tội khủng bố . biết mấy ông thông chốt vào đấy đánh bom thì sao', 'đứa nào cũng có con dù trai hay gái chưa nói trước điều gì . hãy dử cái khẩu nghiệp của mình để sau này hối hận']

2. Số lượng tập Validation: 85 dòng
['vô tới trong nam mà vẫn không tốt lên được à', 'sao không lên cao tốc mà chạy', 'lời nói của người chồng đau lòng người khác nhưng có ai nằm ở vị trí của người vợ không ạ', 'chị vợ ngoại tình cái lồn là sai nhưng phải có gì đó đánh đập chị vợ mới quá sợ không giám về', 'cho bọn ăn cơm này đi đảo hoang']

--- Mẫu dữ liệu đầu tiên trong tập Train ---
Câu văn (Text): gửi clip này cho công an  giao thông điều tra để xử phạt vì phạm pháp luật !
Nhãn (Label):   0 (0=Không ghét, 1=Ghét)

CẤP ĐỘ 2: SAU KHI TOKENIZER (DANH SÁCH SỐ)


In [29]:
# --- PHẦN 2: KIỂM TRA DỮ LIỆU SỐ HÓA (ENCODINGS) ---
print("\n" + "="*30)
print("CẤP ĐỘ 2: SAU KHI TOKENIZER (DANH SÁCH SỐ)")
print("="*30)
first_input_ids = train_encodings['input_ids'][0]


CẤP ĐỘ 2: SAU KHI TOKENIZER (DANH SÁCH SỐ)
